# Track-A prospective paired LOSO residual feasibility v1

Three-fold observed-development experiment. It keeps Track A separate from Track B, trains only on real paired T2-FLAIR data, and evaluates each held-out subject once at the fixed endpoint.

In [ ]:
# GPU preflight before clone/install.
import json, platform, shutil, subprocess
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then reconnect.')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total,compute_cap', '--format=csv,noheader,nounits'], check=True, capture_output=True, text=True).stdout.strip()
if not gpu:
    raise RuntimeError('nvidia-smi found no GPU.')
print(json.dumps({'runtime': 'Google Colab GPU', 'python': platform.python_version(), 'gpu': gpu}, indent=2))


In [ ]:
# Safe same-kernel checkout of the exact pushed experiment commit.
import importlib, sys
from pathlib import Path
REPOSITORY_URL = input('Repository clone URL: ').strip()
EXPECTED_EXPERIMENT_COMMIT = input('Exact LOSO experiment commit SHA: ').strip().lower()
if len(EXPECTED_EXPERIMENT_COMMIT) != 40 or any(c not in '0123456789abcdef' for c in EXPECTED_EXPERIMENT_COMMIT):
    raise ValueError('Enter a full 40-character commit SHA.')
REPO_DIR = Path('/content/MRIxFields')
if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPOSITORY_URL], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone.')
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'cat-file', '-e', EXPECTED_EXPERIMENT_COMMIT + '^{commit}'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', EXPECTED_EXPERIMENT_COMMIT], check=True)
actual = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if actual != EXPECTED_EXPERIMENT_COMMIT:
    raise RuntimeError(f'Checkout mismatch: {actual} != {EXPECTED_EXPERIMENT_COMMIT}')


In [ ]:
# Editable install and same-kernel import repair.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR) + '[dev,nifti]'], check=True)
SOURCE_DIR = str(REPO_DIR / 'src')
if SOURCE_DIR in sys.path:
    sys.path.remove(SOURCE_DIR)
sys.path.insert(0, SOURCE_DIR)
for module_name in list(sys.modules):
    if module_name == 'fieldbridge' or module_name.startswith('fieldbridge.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
import fieldbridge as installed_fieldbridge
PACKAGE_FILE = Path(installed_fieldbridge.__file__).resolve()
if REPO_DIR not in PACKAGE_FILE.parents:
    raise RuntimeError(f'FieldBridge imported from the wrong path: {PACKAGE_FILE}')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot access CUDA.')
print({'commit': actual, 'package': str(PACKAGE_FILE), 'torch': torch.__version__, 'cuda': torch.version.cuda})


In [ ]:
# Private inputs remain external; volumes are copied to ephemeral local scratch.
from google.colab import drive
drive.mount('/content/drive')
MANIFEST_PATH = Path(input('Official Training_prospective JSONL manifest path: ').strip())
SYNTHETIC_CHECKPOINT_PATH = Path(input('Frozen synthetic residual checkpoint path: ').strip())
OUTPUT_DIR = Path(input('Private LOSO output directory: ').strip())
SCRATCH_DIR = Path('/content/fieldbridge_loso_scratch')
for path in (MANIFEST_PATH, SYNTHETIC_CHECKPOINT_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
RUNNER = REPO_DIR / 'notebooks/prospective_paired_loso_residual_runner.py'
CONFIG = REPO_DIR / 'configs/experiment/prospective_paired_loso_residual_v1.yaml'
COMMON = [sys.executable, str(RUNNER), '--manifest', str(MANIFEST_PATH), '--synthetic-checkpoint', str(SYNTHETIC_CHECKPOINT_PATH), '--output-dir', str(OUTPUT_DIR), '--scratch-dir', str(SCRATCH_DIR), '--config', str(CONFIG)]
# Preflight validates manifest multiplicity, geometry, alignment overlays, checkpoint identity, folds, and fixed counts.
subprocess.run([*COMMON, '--preflight'], cwd=REPO_DIR, check=True)
# Dry-run builds both initialization arms and executes finite forward passes without training.
subprocess.run([*COMMON, '--dry-run'], cwd=REPO_DIR, check=True)


In [ ]:
# Fixed-endpoint run. Set RESUME=True only after an interrupted run in the same OUTPUT_DIR.
RESUME = False
command = [*COMMON, *(['--resume'] if RESUME else [])]
subprocess.run(command, cwd=REPO_DIR, check=True)
handoff = json.loads((OUTPUT_DIR / 'sanitized_handoff.json').read_text())
print(json.dumps(handoff, indent=2, sort_keys=True))
